In [ ]:
import json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/icdm_release')

def load(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

mids = [load(f'predictions/medqa/{f}.json')
        for f in ['llama_results','qwen_results','gemma3n_results']]
frontier = load('predictions/medqa/medqa_gpt4o_results.json')

common = set.intersection(*[set(m.keys()) for m in mids]) & set(frontier.keys())
common = {i for i in common if all(m[i]['pred'] is not None for m in mids)
                              and frontier[i]['pred'] is not None}

gpt_correct = sum(1 for i in common if frontier[i]['correct'] == 1)
print(f"=== BASELINE: GPT-4o alone ===")
print(f"  Accuracy: {gpt_correct}/{len(common)} = {gpt_correct/len(common)*100:.2f}%")
print(f"  Cost: 1 frontier call per question")
print(f"  Wrong: {len(common) - gpt_correct} questions ({(1-gpt_correct/len(common))*100:.2f}%)")
print()

maj_correct = 0
maj_wrong_resolved = 0
for i in common:
    preds = [m[i]['pred'] for m in mids]
    from collections import Counter
    maj = Counter(preds).most_common(1)[0][0]
    gold = mids[0][i]['gold']
    if maj == gold:
        maj_correct += 1
print(f"=== BASELINE: Mid-tier majority vote (3 models) ===")
print(f"  Accuracy: {maj_correct}/{len(common)} = {maj_correct/len(common)*100:.2f}%")
print(f"  Cost: 3 mid-tier calls per question")
print()

hits = 0
disagreement_n = confident_n = high_risk_n = 0
for i in common:
    preds = [m[i]['pred'] for m in mids]
    gold = mids[0][i]['gold']
    f_pred = frontier[i]['pred']
    if len(set(preds)) != 1:

        from collections import Counter
        chosen = Counter(preds).most_common(1)[0][0]
        disagreement_n += 1
    elif f_pred == preds[0]:

        chosen = preds[0]
        confident_n += 1
    else:

        chosen = f_pred
        high_risk_n += 1
    if chosen == gold:
        hits += 1

print(f"=== STRATEGY: Detector-routed (mid-tier + escalate to frontier on HIGH_RISK) ===")
print(f"  Accuracy: {hits}/{len(common)} = {hits/len(common)*100:.2f}%")
print(f"  Composition:")
print(f"    CONFIDENT (use mid-tier): {confident_n} questions")
print(f"    DISAGREEMENT (use majority): {disagreement_n} questions")
print(f"    HIGH_RISK (escalate to frontier): {high_risk_n} questions")
print(f"  Cost: 3 mid-tier calls + 1 frontier call per question")
print()

print(f"=== HEAD-TO-HEAD ===")
print(f"  GPT-4o alone:         {gpt_correct/len(common)*100:.2f}% accuracy, cheaper")
print(f"  Mid-tier majority:    {maj_correct/len(common)*100:.2f}% accuracy, cheap")
print(f"  Detector strategy:    {hits/len(common)*100:.2f}% accuracy, costs +1 frontier call vs majority")

In [ ]:
import json
from pathlib import Path
from collections import Counter

DRIVE = Path('/content/drive/MyDrive/icdm_release')
def load(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

mids = [load(f'predictions/medqa/{f}.json')
        for f in ['llama_results','qwen_results','gemma3n_results']]
frontier = load('predictions/medqa/medqa_gpt4o_results.json')

def valid(p):
    return isinstance(p, str) and p in 'ABCD'

common = set.intersection(*[set(m.keys()) for m in mids]) & set(frontier.keys())
common = {i for i in common if all(valid(m[i].get('pred')) for m in mids)
                              and valid(frontier[i].get('pred'))}

print(f"Common questions with valid predictions: {len(common)}\n")

correct_alone = 0
for i in common:
    preds = [m[i]['pred'] for m in mids]
    chosen = Counter(preds).most_common(1)[0][0]
    if chosen == mids[0][i]['gold']:
        correct_alone += 1

correct_with_human = 0
high_risk_n = 0
high_risk_wrong = 0
for i in common:
    preds = [m[i]['pred'] for m in mids]
    gold = mids[0][i]['gold']
    if len(set(preds)) == 1 and frontier[i]['pred'] != preds[0]:
        high_risk_n += 1
        if preds[0] != gold:
            high_risk_wrong += 1
        correct_with_human += 1
    else:
        chosen = Counter(preds).most_common(1)[0][0]
        if chosen == gold:
            correct_with_human += 1

correct_with_escalation = 0
for i in common:
    preds = [m[i]['pred'] for m in mids]
    gold = mids[0][i]['gold']
    if len(set(preds)) == 1 and frontier[i]['pred'] != preds[0]:
        chosen = frontier[i]['pred']
    else:
        chosen = Counter(preds).most_common(1)[0][0]
    if chosen == gold:
        correct_with_escalation += 1

correct_frontier = sum(1 for i in common if frontier[i]['correct'] == 1)

n = len(common)
print(f"=== Scenario A: Mid-tier ensemble alone ===")
print(f"  Accuracy: {correct_alone}/{n} = {correct_alone/n*100:.2f}%")
print(f"  Cost: 3 mid-tier calls, 0% human review")
print()
print(f"=== Scenario B: Ensemble + detector, HIGH_RISK -> human ===")
print(f"  Accuracy (perfect human): {correct_with_human}/{n} = {correct_with_human/n*100:.2f}%")
print(f"  Cost: 3 mid-tier + 1 frontier call per question (for detection)")
print(f"  Human reviews: {high_risk_n}/{n} = {high_risk_n/n*100:.1f}%")
print(f"  HIGH_RISK precision: {high_risk_wrong}/{high_risk_n} = {high_risk_wrong/high_risk_n*100:.1f}%")
print()
print(f"=== Scenario C: Ensemble + detector, HIGH_RISK -> auto-escalate ===")
print(f"  Accuracy: {correct_with_escalation}/{n} = {correct_with_escalation/n*100:.2f}%")
print(f"  Cost: 3 mid-tier + 1 frontier call per question, 0% human review")
print()
print(f"=== Scenario D: Frontier alone ===")
print(f"  Accuracy: {correct_frontier}/{n} = {correct_frontier/n*100:.2f}%")
print(f"  Cost: 1 frontier call")
print()
print(f"=== DELTA ANALYSIS ===")
print(f"  Detector + human improves ensemble by {(correct_with_human - correct_alone)/n*100:+.2f}pp")
print(f"  Detector + auto-escalation improves ensemble by {(correct_with_escalation - correct_alone)/n*100:+.2f}pp")
print(f"  Frontier-only beats best detector strategy by {(correct_frontier - max(correct_with_human, correct_with_escalation))/n*100:+.2f}pp")